# SeaAlert - Notebook 02: Text-to-Speech

This notebook converts the text dataset into clean audio WAV files using **Coqui TTS** (local, no rate limiting).

## Key Features:
- Uses VITS model with 109 different speakers for natural voice variety
- Runs locally - no internet required, no rate limiting
- GPU accelerated when available
- Resume support - can continue from where it left off

## Cell 0 - Setup

In [6]:
# Mount Google Drive (Colab)
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# Install dependencies - Coqui TTS (local, no rate limiting!)
!apt-get -qq update && apt-get -qq install -y ffmpeg espeak-ng
%pip install -q coqui-tts soundfile tqdm

# Imports
import os
import random
import re
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm
from TTS.api import TTS

# Seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Project directory
if IN_COLAB:
    PROJECT_DIR = Path("/content/drive/MyDrive/SeaAlert")
else:
    PROJECT_DIR = Path(".").resolve().parent

# Create folders
AUDIO_CLEAN_DIR = PROJECT_DIR / "data" / "audio_clean"
WAV_DIR = AUDIO_CLEAN_DIR / "wav"
WAV_DIR.mkdir(parents=True, exist_ok=True)

# Configuration
SAMPLE_RATE = 16000  # Standard for speech (Whisper native rate)
MAX_AUDIO = None  # Set to integer for subset (e.g., 100)
CHECKPOINT_EVERY = 50

# TTS Model - VITS with 109 speakers (high quality, local)
TTS_MODEL = "tts_models/en/vctk/vits"

print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"WAV_DIR: {WAV_DIR}")
print(f"TTS Model: {TTS_MODEL}")
print("Using Coqui TTS (local, no rate limiting!)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
PROJECT_DIR: /content/drive/MyDrive/SeaAlert
WAV_DIR: /content/drive/MyDrive/SeaAlert/data/audio_clean/wav
TTS Model: tts_models/en/vctk/vits
Using Coqui TTS (local, no rate limiting!)


## Cell 1 - Load Dataset

In [7]:
# Load dataset
DATASET_PATH = PROJECT_DIR / "data" / "processed" / "02seaalert.csv"

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATASET_PATH}. Run notebook 01 first.")

df = pd.read_csv(DATASET_PATH)
print(f"Loaded {len(df)} samples")

# Select text column to synthesize
TEXT_COLUMN = "text"  # Use original text, not masked

# Subset if needed
if MAX_AUDIO is not None:
    df = df.head(MAX_AUDIO)
    print(f"Subset to {len(df)} samples")

df.head()

Loaded 1872 samples


,idx,text,label,style,scenario_type,has_codeword,codeword,text_masked,vessel,call_sign,mmsi,location,weather,pob,nature,injury
0,0,"MAYDAY, MAYDAY, MAYDAY. This is the fishing ve...",Distress,formal,nav_hazard,True,MAYDAY,"[SIGNAL], [SIGNAL], [SIGNAL]. This is the fish...",Ocean Hunter,WXYZ123,123456789,"37 degrees 45 minutes North, 122 degrees 30 mi...",overcast with a light swell,6,navigation hazard,no injuries reported
1,1,"This is the vessel Ocean Explorer, call sign A...",Safety,formal,medical_issue,False,NONE,"This is the vessel Ocean Explorer, call sign A...",Ocean Explorer,ABCD123,123456789,"34 degrees 15 minutes North, 120 degrees 45 mi...",clear with calm seas,15,medical issue,severe laceration to the arm
2,2,"SECURITE, SECURITE. This is the fishing vessel...",Safety,third_party,steering_failure,True,SECURITE,"[SIGNAL], [SIGNAL]. This is the fishing vessel...",Ocean's Bounty,WZ1234,123456789,"34°12.5'N, 118°28.9'W","Calm, 5 knots from the west",5,steering failure,none
3,3,"This is the vessel Ocean Explorer, call sign A...",Distress,formal,medical_issue,False,NONE,"This is the vessel Ocean Explorer, call sign A...",Ocean Explorer,ABC123,123456789,"34 degrees 15 minutes North, 118 degrees 30 mi...",calm with clear visibility,12,severe allergic reaction,critical condition
4,4,"This is the fishing vessel Ocean Dawn, current...",Distress,formal,water_ingress,False,NONE,"This is the fishing vessel Ocean Dawn, current...",Ocean Dawn,WDC1234,123456789,"34 degrees 15 minutes North, 75 degrees 30 min...","Sea conditions rough, swells 3 meters, winds 3...",5,water ingress in engine room,NaN


## Cell 2 - Initialize TTS Model

In [8]:
# Set TTS cache directory
os.environ["TTS_HOME"] = "/content/tts_cache" if IN_COLAB else str(Path.home() / ".local/share/tts")
Path(os.environ["TTS_HOME"]).mkdir(parents=True, exist_ok=True)

# Clear broken downloads if any
bad_cache = Path(os.environ["TTS_HOME"]) / ".models"
if bad_cache.exists():
    shutil.rmtree(bad_cache, ignore_errors=True)

# Load TTS model
print(f"Loading TTS model: {TTS_MODEL}")
print("This may take a few minutes on first run (downloading model)...")

try:
    tts = TTS(model_name=TTS_MODEL, progress_bar=True, gpu=True)
    print("Using GPU for TTS")
except Exception as e:
    print(f"GPU failed ({e}), using CPU...")
    tts = TTS(model_name=TTS_MODEL, progress_bar=True, gpu=False)
    print("Using CPU for TTS (will be slower)")

# Get available speakers
speakers = getattr(tts, "speakers", None)
print(f"\nModel loaded successfully!")
print(f"Available speakers: {len(speakers) if speakers else 'N/A'}")

Loading TTS model: tts_models/en/vctk/vits
This may take a few minutes on first run (downloading model)...


/usr/local/lib/python3.12/dist-packages/TTS/api.py:93: UserWarning: `gpu` will be deprecated. Please use `tts.to(device)` instead.
  warnings.warn("`gpu` will be deprecated. Please use `tts.to(device)` instead.")


Using GPU for TTS

Model loaded successfully!
Available speakers: 109


## Cell 3 - TTS Functions

In [9]:
def clean_text_for_tts(text):
    """Clean text for better TTS output."""
    if not isinstance(text, str):
        text = str(text)

    # Replace bracketed content with pauses
    text = re.sub(r'\[.*?\]', '...', text)

    # Replace special characters
    text = text.replace('°', ' degrees ')
    text = text.replace('′', ' ')

    # Expand abbreviations
    replacements = {
        r'\bPOB\b': 'persons on board',
        r'\bVHF\b': 'V H F',
        r'\bGMDSS\b': 'G M D S S',
        r'\bMMSI\b': 'M M S I',
        r'\bETA\b': 'E T A',
        r'\bM/V\b': 'motor vessel',
        r'\bMV\b': 'motor vessel',
    }

    for pattern, replacement in replacements.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)

    # Remove excessive whitespace
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()

    # Limit length (for very long texts)
    if len(text) > 500:
        text = text[:500]

    return text

def text_to_wav(text, output_path, speaker=None):
    """Convert text to WAV file using Coqui TTS."""
    try:
        # Generate speech
        wav = np.array(tts.tts(text=text, speaker=speaker), dtype=np.float32)

        # Save to file
        sf.write(str(output_path), wav, SAMPLE_RATE)

        return True, None

    except Exception as e:
        return False, str(e)[:100]

# Test TTS
print("Testing Coqui TTS...")
test_text = "MAYDAY MAYDAY MAYDAY. This is motor vessel Blue Horizon."
test_clean = clean_text_for_tts(test_text)
print(f"Original: {test_text}")
print(f"Cleaned: {test_clean}")

# Select a random speaker for test
test_speaker = np.random.choice(speakers) if speakers else None
print(f"Test speaker: {test_speaker}")

test_path = WAV_DIR / "test.wav"
success, error = text_to_wav(test_clean, test_path, speaker=test_speaker)
if success:
    info = sf.info(str(test_path))
    print(f"Test file saved: {test_path} ({info.duration:.2f}s)")
    # Clean up
    if test_path.exists():
        test_path.unlink()
else:
    print(f"Test failed: {error}")

Testing Coqui TTS...
Original: MAYDAY MAYDAY MAYDAY. This is motor vessel Blue Horizon.
Cleaned: MAYDAY MAYDAY MAYDAY. This is motor vessel Blue Horizon.
Test speaker: p360
Test file saved: /content/drive/MyDrive/SeaAlert/data/audio_clean/wav/test.wav (5.63s)


## Cell 4 - Generate WAVs (Local, No Rate Limiting!)

In [10]:
# Output paths
AUDIO_INDEX_PATH = AUDIO_CLEAN_DIR / "audio_index.csv"

# Check for existing progress (RESUME FROM CHECKPOINT)
existing_indices = set()
results = []

if AUDIO_INDEX_PATH.exists():
    existing_df = pd.read_csv(AUDIO_INDEX_PATH)
    existing_indices = set(existing_df["idx"].tolist())
    results = existing_df.to_dict('records')
    print(f"Resuming from checkpoint: {len(results)} files already generated")

# Estimate time (much faster without rate limiting!)
remaining = len(df) - len(existing_indices)
est_seconds = remaining * 1.5  # ~1.5 second per sample with Coqui TTS
print(f"Samples to process: {remaining}")
print(f"Estimated time: {est_seconds/60:.1f} minutes ({est_seconds/3600:.1f} hours)")
print(f"\nNote: Using Coqui TTS (local) - NO rate limiting! Much faster than gTTS.")

# Synthesize audio
pbar = tqdm(df.iterrows(), total=len(df), desc="Synthesizing audio")
errors = []

for i, row in pbar:
    idx = row["idx"]

    # Skip if already processed
    if idx in existing_indices:
        continue

    text = row[TEXT_COLUMN]
    text_clean = clean_text_for_tts(text)
    wav_path = WAV_DIR / f"{idx}.wav"

    # Select random speaker for variety
    speaker = np.random.choice(speakers) if speakers else None

    # Generate audio (no retry needed - local processing!)
    success, error = text_to_wav(text_clean, wav_path, speaker=speaker)

    if success:
        results.append({
            "idx": idx,
            "wav_path_clean": str(wav_path),
            "label": row.get("label", ""),
            "style": row.get("style", ""),
            "scenario_type": row.get("scenario_type", ""),
            "speaker": speaker,
        })
    else:
        errors.append({"idx": idx, "error": error})

    # Checkpoint save
    if len(results) % CHECKPOINT_EVERY == 0 and len(results) > 0:
        pd.DataFrame(results).to_csv(AUDIO_INDEX_PATH, index=False)
        pbar.set_postfix({"saved": len(results), "errors": len(errors)})

# Final save
audio_index_df = pd.DataFrame(results)
audio_index_df.to_csv(AUDIO_INDEX_PATH, index=False)

print(f"\nGenerated {len(results)} audio files")
print(f"Errors: {len(errors)}")
if errors:
    print(f"First errors: {errors[:3]}")
print(f"Audio index saved to: {AUDIO_INDEX_PATH}")

Resuming from checkpoint: 125 files already generated
Samples to process: 1747
Estimated time: 43.7 minutes (0.7 hours)

Note: Using Coqui TTS (local) - NO rate limiting! Much faster than gTTS.


Synthesizing audio:   0%|          | 0/1872 [00:00<?, ?it/s]


Generated 1872 audio files
Errors: 0
Audio index saved to: /content/drive/MyDrive/SeaAlert/data/audio_clean/audio_index.csv


## Cell 5 - Sanity Check

In [11]:
# Verify audio files
audio_index_df = pd.read_csv(AUDIO_INDEX_PATH)

print(f"Total audio files in index: {len(audio_index_df)}")

# Check file existence
exists_count = 0
missing_files = []

for _, row in audio_index_df.iterrows():
    wav_path = Path(row["wav_path_clean"])
    if wav_path.exists():
        exists_count += 1
    else:
        missing_files.append(str(wav_path))

print(f"Files exist: {exists_count}/{len(audio_index_df)}")
if missing_files:
    print(f"Missing files: {missing_files[:5]}...")

# Sample file info
print("\n5 random audio files:")
for _, row in audio_index_df.sample(min(5, len(audio_index_df)), random_state=SEED).iterrows():
    wav_path = Path(row["wav_path_clean"])
    if wav_path.exists():
        info = sf.info(str(wav_path))
        print(f"  {wav_path.name}: {info.duration:.2f}s, {info.samplerate}Hz, speaker={row.get('speaker', 'N/A')}")

# Speaker distribution
if "speaker" in audio_index_df.columns:
    print(f"\nSpeaker distribution:")
    print(f"  Unique speakers used: {audio_index_df['speaker'].nunique()}")

# Play sample audio (Colab only)
if IN_COLAB and len(audio_index_df) > 0:
    from IPython.display import Audio, display

    sample_row = audio_index_df.iloc[0]
    sample_path = sample_row["wav_path_clean"]
    sample_idx = sample_row["idx"]

    # Get original text
    orig_text = df[df["idx"] == sample_idx]["text"].values
    if len(orig_text) > 0:
        print(f"\nSample audio (idx={sample_idx}):")
        print(f"Text: {orig_text[0][:100]}...")
        display(Audio(sample_path))

Total audio files in index: 1872
Files exist: 1872/1872

5 random audio files:
  1703.wav: 49.88s, 16000Hz, speaker=p376
  1225.wav: 46.50s, 16000Hz, speaker=p229
  1460.wav: 37.77s, 16000Hz, speaker=p364
  275.wav: 43.05s, 16000Hz, speaker=p343
  416.wav: 41.69s, 16000Hz, speaker=p244

Speaker distribution:
  Unique speakers used: 109

Sample audio (idx=0):
Text: MAYDAY, MAYDAY, MAYDAY. This is the fishing vessel Ocean Hunter, call sign WXYZ123, MMSI 123456789. ...


In [12]:
# Coverage check
total_needed = len(df)
total_generated = len(audio_index_df)
coverage = 100 * total_generated / total_needed

print("\n" + "=" * 60)
print("TTS GENERATION STATUS")
print("=" * 60)
print(f"\nCoverage: {total_generated}/{total_needed} ({coverage:.1f}%)")

if coverage < 100:
    print(f"\nMissing: {total_needed - total_generated} samples")
    print("\nTo generate remaining files:")
    print("1. Simply re-run Cell 4 - it will resume from checkpoint")
else:
    print("\nAll samples generated successfully!")

print(f"\nArtifacts:")
print(f"  1. {WAV_DIR}/*.wav ({total_generated} files)")
print(f"  2. {AUDIO_INDEX_PATH}")


TTS GENERATION STATUS

Coverage: 1872/1872 (100.0%)

All samples generated successfully!

Artifacts:
  1. /content/drive/MyDrive/SeaAlert/data/audio_clean/wav/*.wav (1872 files)
  2. /content/drive/MyDrive/SeaAlert/data/audio_clean/audio_index.csv
